# 计算 $n_{0.5}$ 和 $n_{1.0}$

In [ ]:
# 计算 $n_{0.5}$ 和 $n_{1.0}$, 并保存到 CSV 文件中, 以便后续与化学元素浓度进行相关性分析

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

final_psd_df = pd.read_csv(r"D:\Coding\Data\Lanzhou_aerosol\SMPS+APS\final_psd(v1.0.2).csv", index_col=0, parse_dates=True)
status_df = pd.read_csv(r"D:\Coding\Data\Lanzhou_aerosol\SMPS+APS\instrument_status(v1.0.2).csv", index_col=0, parse_dates=True)

mask = status_df['status_aps'] == True
df = final_psd_df[mask]

# 1. 提取数值格式的列名
diameters = df.columns.astype(float)

# 2. 计算每个通道的对数区间宽度 dlogDp
# np.gradient 会自动使用中心差分计算中间点，单边差分计算边界点，非常适合这个场景
dlogDp = np.gradient(np.log10(diameters))

# 3. 计算每个通道的实际数浓度 (dN)
# 公式: dN = (dN/dlogDp) * dlogDp
delta_N_df = df * dlogDp

# 4. 定义你想计算的粒径范围列表 [(下限, 上限), ...]
size_ranges = [
    (10, 500),
    (10, 1000),
    (500, 2500),
    (1000, 2500),
]

results = {}

for D_low, D_high in size_ranges:
    # 找出落在该范围内的所有列名（布尔掩码）
    # 注意：这里使用 >= 和 <=，如果原数据列名没有精确到整数（比如2491而不是2500），
    # 这个掩码会自动把小于等于上限的列都包含进去
    mask = (diameters >= D_low) & (diameters <= D_high)
    
    # 提取这些列的实际浓度，并沿行方向（axis=1）求和
    col_name = f'N_{D_low}-{D_high}nm'
    results[col_name] = delta_N_df.loc[:, mask].sum(axis=1)

# 将结果合并为一个新的 DataFrame
number_conc_df = pd.DataFrame(results)

# 打印结果看看前几行
print(number_conc_df.head())

df_element = pd.read_csv(r"D:\Coding\Data\Lanzhou_chemical\Corr(INP_vs_element).csv").drop(columns=['datetime'])
df_element['Time'] = pd.to_datetime(df_element['Time'])
result = pd.merge_asof(
	df_element,
	number_conc_df,
	left_on="Time",
	right_on="__dt__",
	direction="nearest",
	tolerance=pd.Timedelta("1h")
)

result.to_csv(r"D:\Coding\Data\Lanzhou_chemical\Corr_heatmap.csv", index=False) # 目前计算有误, 数据不能用

                      N_10-500nm  N_10-1000nm  N_500-2500nm  N_1000-2500nm
__dt__                                                                    
2024-12-06 13:40:00  7596.316010  7624.750994     29.983530       1.548546
2024-12-06 13:50:00  6758.152787  6783.265361     26.504323       1.391748
2024-12-06 14:00:00  6022.684191  6045.382780     24.012315       1.313726
2024-12-06 14:10:00  5770.049937  5791.277252     22.459027       1.231712
2024-12-06 14:20:00  6152.166092  6174.188128     23.241595       1.219558
